In [2]:
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query

In [3]:
run_query("""
    SELECT TOP 2 *
    FROM week06.london_2014
""")

,GMT,Max TemperatureC,Mean TemperatureC,Min TemperatureC,Dew PointC,MeanDew PointC,Min DewpointC,Max Humidity,Mean Humidity,Min Humidity,...,Max VisibilityKm,Mean VisibilityKm,Min VisibilitykM,Max Wind SpeedKm/h,Mean Wind SpeedKm/h,Max Gust SpeedKm/h,Precipitationmm,CloudCover,Events,WindDirDegrees<br />
0,2014-1-1,11,8,6,9,7,4,94,86,73,...,31,11,2,40,26,66.0,9.91,4.0,Rain,186<br />
1,2014-1-2,11,9,6,9,6,4,94,81,60,...,31,15,5,32,21,50.0,1.02,2.0,Rain,214<br />


In [14]:
run_query("""
    SELECT *
    FROM week19.wine
""")

,Alcohol,Malic_Acid,Ash,Ash_Alcanity,Magnesium,Total_Phenols,Flavanoids,Nonflavanoid_Phenols,Proanthocyanins,Color_Intensity,Hue,OD280,Proline,Customer_Segment
0,12.19,2.81,2.35,20.8,97,1.75,1.60,0.30,0.40,2.35,1.03,2.71,479,2
1,12.55,1.81,2.11,22.3,86,1.85,0.80,0.39,1.48,2.75,1.06,2.44,557,2
2,12.96,0.70,2.54,17.2,108,2.92,3.23,0.29,1.98,5.88,1.23,2.90,1272,1
3,13.24,1.92,2.50,14.8,95,2.88,3.50,0.22,1.52,5.29,1.11,2.89,1048,1
4,13.20,1.56,2.41,20.1,96,1.20,0.96,0.40,0.73,10.08,0.60,1.20,861,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,12.85,3.62,2.22,19.6,96,1.90,0.59,0.49,1.95,6.67,0.91,1.20,757,3
174,12.72,2.32,2.41,16.7,99,2.21,2.03,0.50,1.39,2.10,0.82,2.64,734,2
175,13.37,0.70,2.35,17.9,111,3.05,2.78,0.27,2.32,5.30,0.65,3.67,899,1
176,13.33,1.42,2.69,21.9,97,2.31,0.78,0.59,2.45,7.69,0.76,1.39,934,3


In [ ]:
cities = run_query("""
    SELECT 'London' AS city, GMT, [Mean TemperatureC] FROM week06.london_2014
    UNION ALL SELECT 'Beijing',  Date, [Mean TemperatureC] FROM week06.beijing_pek_2014
    UNION ALL SELECT 'Brasilia', Date, [Mean TemperatureC] FROM week06.brasilia_bsb_2014
    UNION ALL SELECT 'CapeTown', Date, [Mean TemperatureC] FROM week06.capetown_cpt_2014
    UNION ALL SELECT 'Delhi',    Date, [Mean TemperatureC] FROM week06.delhi_del_2014
    UNION ALL SELECT 'Moscow',   Date, [Mean TemperatureC] FROM week06.moscow_svo_2014
""")
cities.shape

In [ ]:
avg_temp_by_cities = run_query("""
    SELECT 'London' AS city, AVG([Mean TemperatureC]) as Avg_temp FROM week06.london_2014
    UNION ALL SELECT 'Beijing',  AVG([Mean TemperatureC]) FROM week06.beijing_pek_2014
    UNION ALL SELECT 'Brasilia', AVG([Mean TemperatureC]) FROM week06.brasilia_bsb_2014
    UNION ALL SELECT 'CapeTown', AVG([Mean TemperatureC]) FROM week06.capetown_cpt_2014
    UNION ALL SELECT 'Delhi',    AVG([Mean TemperatureC]) FROM week06.delhi_del_2014
    UNION ALL SELECT 'Moscow',   AVG([Mean TemperatureC]) FROM week06.moscow_svo_2014
""")
avg_temp_by_cities.head()

In [3]:
# --- Connect to the cohort database (read-only) ---
import sys, pathlib, re
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query, get_engine, get_connection

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

_BLOCKED = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|EXEC|EXECUTE|GRANT|REVOKE|DENY|INTO|BACKUP|RESTORE)\b",
    re.IGNORECASE,
)

def q(sql: str) -> pd.DataFrame:
    """Run a read-only query and return a DataFrame.

    String literals and comments are stripped before the safety check, so a value
    like 'Delete me' inside quotes is fine.
    """
    scrubbed = re.sub(r"'([^']|'')*'", "''", sql)           # string literals
    scrubbed = re.sub(r"--[^\n]*", "", scrubbed)              # -- comments
    scrubbed = re.sub(r"/\*.*?\*/", "", scrubbed, flags=re.S) # /* comments */
    hit = _BLOCKED.search(scrubbed)
    if hit:
        raise PermissionError(
            f"'{hit.group(0).upper()}' is not allowed: this notebook is read-only. "
            "Shape your data inside the SELECT, or in pandas."
        )
    return run_query(sql)

q("SELECT DB_NAME() AS database_name, CURRENT_USER AS logged_in_as, SYSDATETIME() AS server_time")

,database_name,logged_in_as,server_time
0,assessmentDB,intern_readonly,2026-10-15 21:00:39.891252


In [ ]:
try:
    q("DELETE FROM week06.london_2014")
except PermissionError as e:
    print("Blocked as expected ->", e)

In [ ]:
conn = get_connection()
try:
    cur = conn.cursor()
    cur.execute("UPDATE week06.london_2014 SET [Mean TemperatureC] = 0 WHERE 1 = 0")
    print("Unexpected: the server accepted the statement. Tell the admin the account isn't read-only.")
except Exception as e:
    print("Server refused, as expected ->", str(e)[:160])
finally:
    conn.rollback()
    conn.close()

In [ ]:
london = q("""
    SELECT *
    FROM week06.london_2014
""")
london.head(2)

In [85]:
tables = q("""
    SELECT *
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
""")
print(len(tables), "tables")
tables.head(5)

67 tables


,TABLE_CATALOG,TABLE_SCHEMA,TABLE_NAME,TABLE_TYPE
0,assessmentDB,sys,database_firewall_rules,VIEW
1,assessmentDB,week06,beijing_pek_2014,BASE TABLE
2,assessmentDB,week06,brasilia_bsb_2014,BASE TABLE
3,assessmentDB,week06,capetown_cpt_2014,BASE TABLE
4,assessmentDB,week06,delhi_del_2014,BASE TABLE


In [87]:
columns = q("""
    SELECT ORDINAL_POSITION, COLUMN_NAME, DATA_TYPE, IS_NULLABLE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = 'beijing_pek_2014'
    ORDER BY ORDINAL_POSITION DESC
""")
print(columns.head(3))

   ORDINAL_POSITION           COLUMN_NAME DATA_TYPE IS_NULLABLE
0                23  WindDirDegrees<br />   varchar         YES
1                22                Events   varchar         YES
2                21            CloudCover     float         YES


In [88]:
q("""
    SELECT TABLE_SCHEMA + '.' + TABLE_NAME AS table_name, COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE COLUMN_NAME LIKE '%Income%' OR COLUMN_NAME = 'Age'
    ORDER BY table_name
""")

,table_name,COLUMN_NAME,DATA_TYPE
0,week14.assignment1_ecommerce_data,Age,float
1,week14.assignment1_ecommerce_data,Annual_Income,float
2,week14.data,Age,float
3,week14.housing_price_data,Age,bigint
4,week14.task1_data_with_missing,Age,float
5,week14.task1_data_with_missing,Income,float
6,week14.task2_categorical_data,Age,bigint
7,week14.task3_scaling_data,Age,bigint
8,week16.assignment1_house_prices,Age,bigint
9,week16.assignment2_marketing_campaign,Age,bigint


In [83]:
q("""
    SELECT
        COUNT(*)                                   AS n_rows,
        COUNT(DISTINCT Gender)                     AS n_gender_values,
        SUM(CASE WHEN Gender IS NULL THEN 1 ELSE 0 END)           AS missing_gender,
        SUM(CASE WHEN Applicant_Income IS NULL THEN 1 ELSE 0 END) AS missing_income,
        SUM(CASE WHEN Loan_Amount IS NULL THEN 1 ELSE 0 END)      AS missing_loan_amount,
        MIN(Applicant_Income)                      AS min_income,
        MAX(Applicant_Income)                      AS max_income
    FROM week17.loan_approval_prediction
    """)

,n_rows,n_gender_values,missing_gender,missing_income,missing_loan_amount,min_income,max_income
0,500,2,0,0,0,17000,80000


In [98]:
q("""
    SELECT TABLE_SCHEMA + '_' + TABLE_NAME AS Table_Name, COUNT(*) AS n_column
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA IN ('week16', 'week17')
    GROUP BY TABLE_SCHEMA, TABLE_NAME
    ORDER BY n_column DESC
""")

,Table_Name,n_column
0,week16_customer_churn_prediction,19
1,week17_loan_approval_prediction,16
2,week16_assignment3_credit_risk,10
3,week17_assignment2_medical_diagnosis,10
4,week16_assignment2_marketing_campaign,9
5,week16_task3_classification_data,9
6,week16_assignment1_house_prices,8
7,week17_assignment1_product_recommendation,8
8,week17_assignment3_classifier_comparison,7
9,week17_task3_naive_bayes_data,7


In [107]:
q("""
    SELECT GMT, [Max TemperatureC]
    FROM week06.london_2014
    ORDER BY [Max TemperatureC] DESC
    OFFSET 10 ROWS FETCH NEXT 5 ROWS ONLY
""")

,GMT,Max TemperatureC
0,2014-7-20,27
1,2014-7-21,27
2,2014-7-22,27
3,2014-7-25,27
4,2014-7-29,27


In [17]:
q("""
    SELECT TABLE_NAME as table_name, count(*) as n_colum
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'week06'
    GROUP BY table_name
    ORDER BY table_name
""")

,table_name,n_colum
0,beijing_pek_2014,23
1,brasilia_bsb_2014,23
2,capetown_cpt_2014,23
3,delhi_del_2014,23
4,london_2014,23
5,moscow_svo_2014,23
6,who_pop_tb_all,3
7,who_pop_tb_some,3


In [42]:
table_whitespaced_in_londontable= q("""
    SELECT COLUMN_NAME
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = 'london_2014' AND 
                        TABLE_SCHEMA = 'week06' AND 
                        COLUMN_NAME LIKE ' %'
""")
print(table_whitespaced_in_londontable)

                    COLUMN_NAME
0                    CloudCover
1                        Events
2            Max Gust SpeedKm/h
3     Max Sea Level PressurehPa
4              Max VisibilityKm
5            Max Wind SpeedKm/h
6                 Mean Humidity
7    Mean Sea Level PressurehPa
8             Mean VisibilityKm
9           Mean Wind SpeedKm/h
10                 Min Humidity
11    Min Sea Level PressurehPa
12             Min VisibilitykM


In [65]:
q("""
    SELECT SUM(CASE WHEN [ Events] IS NULL THEN 1 ELSE 0 END) as no_event_days,
           SUM(CASE WHEN [ Events] LIKE '%Rain%' THEN 1 ELSE 0 END) as rain_days,
           SUM(CASE WHEN [ Events] IS NULL AND Precipitationmm > 0 THEN 1 ELSE 0 END) as wet_but_no_event_days
    FROM week06.london_2014
""")

,no_event_days,rain_days,wet_but_no_event_days
0,114,238,6


In [84]:
q("""
    SELECT Contract_Type, 
           Count(*) as customers, 
           SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END) as churned,
           ROUND(SUM(CASE WHEN Churn = 1 THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) as churn_rate_pct
    FROM week16.customer_churn_prediction
    GROUP BY Contract_Type
    ORDER BY churn_rate_pct DESC
""")

,Contract_Type,customers,churned,churn_rate_pct
0,Month-to-Month,158,91,57.60
1,One Year,165,89,53.90
2,Two Year,177,48,27.10


In [ ]:
q("""
    SELECT TOP 2 DISTINCT*
    FROM week09.comtrade_milk_uk_monthly_14
""")

,Classification,Year,Period,Period Desc.,Aggregate Level,Is Leaf Code,Trade Flow Code,Trade Flow,Reporter Code,Reporter,Reporter ISO,Partner Code,Partner,Partner ISO,2nd Partner Code,2nd Partner,2nd Partner ISO,Customs Proc. Code,Customs,Mode of Transport Code,Mode of Transport,Commodity Code,Commodity,Qty Unit Code,Qty Unit,Qty,Alt Qty Unit Code,Alt Qty Unit,Alt Qty,Netweight (kg),Gross weight (kg),Trade Value (US$),CIF Trade Value (US$),FOB Trade Value (US$),Flag
0,HS,2014,201401,January 2014,4,0,1,Imports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,22404316,None,21950747,None,None,0
1,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,60497363,None,46923551,None,None,0


In [108]:
q("""
    SELECT Partner,
           SUM([Trade Value (US$)]) as total_exports_usd 
    FROM week09.comtrade_milk_uk_monthly_14
    WHERE [Trade Flow] = 'Exports' AND 
          Partner <> 'World'
    GROUP BY Partner
    HAVING COUNT(DISTINCT [Period Desc.]) = 12
    ORDER BY total_exports_usd DESC
""")

,Partner,total_exports_usd
0,Ireland,421228441
1,Algeria,93753841
2,Netherlands,68649082
3,Germany,36096633
4,France,34622982
5,Belgium,29343007
6,China,27317861
7,"China, Hong Kong SAR",19847947
8,Nigeria,12932371
9,Guinea,12018176


In [129]:
q("""
    SELECT b.country AS Country,
           ROUND([TB deaths]* 100000/[SP.POP.TOTL], 1) AS tb_deaths_per_100k
    FROM week06.who_pop_tb_all as a
    INNER JOIN week08.wb_pop_2013 as b
        ON a.country = b.country
""")

,Country,tb_deaths_per_100k
0,Afghanistan,42.40
1,Albania,0.70
2,Algeria,13.40
3,Andorra,0.30
4,Angola,29.40
...,...,...
161,Uruguay,1.20
162,Uzbekistan,7.30
163,Vanuatu,6.30
164,Zambia,23.60


In [137]:
q("""
    SELECT TOP 10 *
    FROM week06.who_pop_tb_all
    ORDER BY Country DESC 
""")

,Country,Population (1000s),TB deaths
0,Zimbabwe,14150,"5,700.00"
1,Zambia,14539,"3,600.00"
2,Yemen,24407,990.00
3,Viet Nam,91680,"17,000.00"
4,Venezuela (Bolivarian Republic of),30405,480.00
5,Vanuatu,253,16.00
6,Uzbekistan,28934,"2,200.00"
7,Uruguay,3407,40.00
8,United States of America,320051,490.00
9,United Republic of Tanzania,49253,"6,000.00"


In [143]:
q("""
    SELECT DISTINCT Country
    FROM week06.who_pop_tb_all
    WHERE Country LIKE '%Bolivia%'
""")

,Country
0,Bolivia (Plurinational State of)


In [154]:
q("""
    SELECT *
    FROM week06.who_pop_tb_all AS a
""")

,Country,Population (1000s),TB deaths
0,Afghanistan,30552,"13,000.00"
1,Albania,3173,20.00
2,Algeria,39208,"5,100.00"
3,Andorra,79,0.26
4,Angola,21472,"6,900.00"
...,...,...,...
189,Venezuela (Bolivarian Republic of),30405,480.00
190,Viet Nam,91680,"17,000.00"
191,Yemen,24407,990.00
192,Zambia,14539,"3,600.00"


In [164]:
q("""
    SELECT 'Beijing' AS city,  ROUND(AVG(1.0* [Mean TemperatureC]), 1) AS avg_mean_temp_c FROM week06.beijing_pek_2014
    UNION ALL SELECT 'Brasilia', ROUND(AVG(1.0*[Mean TemperatureC]), 1) FROM week06.brasilia_bsb_2014
    UNION ALL SELECT 'CapeTown', ROUND(AVG(1.0*[Mean TemperatureC]), 1) FROM week06.capetown_cpt_2014
    UNION ALL SELECT 'Delhi',    ROUND(AVG(1.0*[Mean TemperatureC]), 1) FROM week06.delhi_del_2014
    UNION ALL SELECT 'London', ROUND(AVG(1.0*[Mean TemperatureC]), 1) as Avg_temp FROM week06.london_2014
    UNION ALL SELECT 'Moscow',   ROUND(AVG(1.0*[Mean TemperatureC]), 1) FROM week06.moscow_svo_2014
    ORDER BY city
""")

,city,avg_mean_temp_c
0,Beijing,13.40
1,Brasilia,22.90
2,CapeTown,17.60
3,Delhi,13.70
4,London,12.30
5,Moscow,6.00


In [165]:
q("""
    SELECT TOP 5
        GMT                                  AS raw_value,
        TRY_CONVERT(date, GMT)               AS as_date,
        DATENAME(weekday, TRY_CONVERT(date, GMT)) AS day_name,
        MONTH(TRY_CONVERT(date, GMT))        AS month_no,
        DATENAME(month, TRY_CONVERT(date, GMT))   AS month_name,
        DATEPART(quarter, TRY_CONVERT(date, GMT)) AS qtr,
        EOMONTH(TRY_CONVERT(date, GMT))      AS month_end
    FROM week06.london_2014
""")

,raw_value,as_date,day_name,month_no,month_name,qtr,month_end
0,2014-1-1,2014-01-01,Wednesday,1,January,1,2014-01-31
1,2014-1-2,2014-01-02,Thursday,1,January,1,2014-01-31
2,2014-1-3,2014-01-03,Friday,1,January,1,2014-01-31
3,2014-1-4,2014-01-04,Saturday,1,January,1,2014-01-31
4,2014-1-5,2014-01-05,Sunday,1,January,1,2014-01-31


In [31]:
q("""
    WITH practice AS (SELECT TRY_CONVERT(date, GMT) AS Date,
                             MONTH(TRY_CONVERT(date, GMT)) AS month_no,
                             [Max TemperatureC] AS Temperature,
                             ROW_NUMBER() OVER (
                                PARTITION BY MONTH(TRY_CONVERT(date, GMT))
                                ORDER BY [Max TemperatureC] DESC, TRY_CONVERT(date, GMT) ASC
                                ) AS Rank_num
                        FROM week06.london_2014)
    SELECT month_no, Date, Rank_num
    FROM practice
    WHERE Rank_num = 1
""")

,month_no,Date,Rank_num
0,1,2014-01-05,1
1,2,2014-02-24,1
2,3,2014-03-09,1
3,4,2014-04-01,1
4,5,2014-05-19,1
5,6,2014-06-09,1
6,7,2014-07-18,1
7,8,2014-08-06,1
8,9,2014-09-18,1
9,10,2014-10-31,1


In [17]:
london_daily = q("""
    SELECT TRY_CONVERT(date, GMT) AS day,
           [Mean TemperatureC]    AS mean_temp,
           Precipitationmm        AS rain_mm,
           SUM(Precipitationmm) OVER (ORDER BY TRY_CONVERT(date, GMT)
                                      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS rain_ytd_mm,
           AVG(1.0 * [Mean TemperatureC]) OVER (ORDER BY TRY_CONVERT(date, GMT)
                                      ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)         AS temp_7d_avg
    FROM week06.london_2014
    ORDER BY day
""")
london_daily.head(10)

,day,mean_temp,rain_mm,rain_ytd_mm,temp_7d_avg
0,2014-01-01,8,9.91,9.91,8.00
1,2014-01-02,9,1.02,10.93,8.50
2,2014-01-03,8,7.11,18.04,8.33
3,2014-01-04,6,9.91,27.95,7.75
4,2014-01-05,7,0.25,28.20,7.60
5,2014-01-06,10,2.03,30.23,8.00
6,2014-01-07,10,5.08,35.31,8.29
7,2014-01-08,9,1.02,36.33,8.43
8,2014-01-09,8,0.00,36.33,8.29
9,2014-01-10,7,0.25,36.58,8.14


In [32]:
q("""
    SELECT TOP 2 *
    FROM week09.comtrade_milk_uk_monthly_14
""")

,Classification,Year,Period,Period Desc.,Aggregate Level,Is Leaf Code,Trade Flow Code,Trade Flow,Reporter Code,Reporter,Reporter ISO,Partner Code,Partner,Partner ISO,2nd Partner Code,2nd Partner,2nd Partner ISO,Customs Proc. Code,Customs,Mode of Transport Code,Mode of Transport,Commodity Code,Commodity,Qty Unit Code,Qty Unit,Qty,Alt Qty Unit Code,Alt Qty Unit,Alt Qty,Netweight (kg),Gross weight (kg),Trade Value (US$),CIF Trade Value (US$),FOB Trade Value (US$),Flag
0,HS,2014,201401,January 2014,4,0,1,Imports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,22404316,None,21950747,None,None,0
1,HS,2014,201401,January 2014,4,0,2,Exports,826,United Kingdom,None,0,World,None,None,None,None,None,None,None,None,0401,"Milk and cream, neither concentrated nor sweet...",0,None,None,None,None,None,60497363,None,46923551,None,None,0


In [63]:
q("""
    WITH milk_transact_without_world AS (SELECT *
                                    FROM week09.comtrade_milk_uk_monthly_14
                                    WHERE Partner <> 'World')
    SELECT Period,
            SUM([Trade Value (US$)]) OVER (PARTITION BY Period) AS exports_usd,
            SUM([Trade Value (US$)]) OVER (ORDER BY Period
                                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ytd_exports_usd
    FROM milk_transact_without_world
""")

,Period,exports_usd,ytd_exports_usd
0,201401,139512730,3410
1,201401,139512730,3726
2,201401,139512730,4476075
3,201401,139512730,10139203
4,201401,139512730,10173769
...,...,...,...
1424,201412,95273664,1462374356
1425,201412,95273664,1462376928
1426,201412,95273664,1462410889
1427,201412,95273664,1462422434


In [52]:
q("""
 WITH without_world AS (SELECT Period,
                                SUM([Trade Value (US$)]) OVER (PARTITION BY Period) AS exports_usd,
                                SUM([Trade Value (US$)]) OVER (ORDER BY Period
                                                                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS ytd_exports_usd
                                FROM week09.comtrade_milk_uk_monthly_14
                                WHERE Partner <> 'World')
        SELECT Period, MAX(exports_usd) AS exports_usd, MAX(ytd_exports_usd) AS ytd_exports_usd
        FROM without_world
        GROUP BY Period
""")

,Period,exports_usd,ytd_exports_usd
0,201401,139512730,139512730
1,201402,111789783,251302513
2,201403,131100878,382403391
3,201404,139406385,521809776
4,201405,144276259,666086035
5,201406,144420204,810506239
6,201407,117675467,928181706
7,201408,102032536,1030214242
8,201409,117302838,1147517080
9,201410,113368960,1260886040


### SQL Task 10 · Business question
The Head of Credit asks: *"How does the approval rate vary by property area and credit history?"*
Using `week17.loan_approval_prediction`, return one row per `Property_Area` × `Credit_History` combination
that exists in the data, with the number of `applications` and the `approval_rate_pct` (rounded to 1 decimal).

**Return:** `Property_Area`, `Credit_History`, `applications`, `approval_rate_pct`, sorted by `Property_Area`
then `Credit_History`. Then answer reflection question 3 below about what you see.

In [64]:
q("""
    SELECT DISTINCT Loan_Approved
    FROM week17.loan_approval_prediction
""")

,Loan_Approved
0,0
1,1


In [ ]:
q("""
    SELECT Property_Area, Credit_History,
    DENSE_RANK() OVER (ORDER BY Property_Area, Credit_History) AS Rank,
    COUNT(*) OVER (PARTITION )
    FROM week17.loan_approval_prediction
""")

,Property_Area,Credit_History,Rank
0,Rural,0,1
1,Rural,0,1
2,Rural,0,1
3,Rural,0,1
4,Rural,0,1
...,...,...,...
495,Urban,1,3
496,Urban,1,3
497,Urban,1,3
498,Urban,1,3


In [74]:
q("""
    SELECT Property_Area, 
        Credit_History, 
        COUNT(*) AS Applications,
        ROUND(100.0 * SUM(CASE WHEN Loan_Approved = 1 THEN 1 ELSE 0 END)/COUNT(*), 1) AS approved_rate_pct
    FROM week17.loan_approval_prediction
    GROUP BY Property_Area, Credit_History
    ORDER BY Property_Area, Credit_History
""")

,Property_Area,Credit_History,Applications,approved_rate_pct
0,Rural,0,166,0.00
1,Semiurban,1,167,100.00
2,Urban,1,167,100.00


In [72]:
q("""
    SELECT Property_Area, Loan_Approved
    FROM week17.loan_approval_prediction
    WHERE Property_Area = 'Rural' AND Loan_Approved = 1
""")

,Property_Area,Loan_Approved


In [76]:
q("""
    SELECT 'Beijing' AS city,  AVG([Mean TemperatureC]) AS avg_mean_temp_c FROM week06.beijing_pek_2014
    UNION ALL SELECT 'Brasilia', AVG([Mean TemperatureC]) FROM week06.brasilia_bsb_2014
    UNION ALL SELECT 'Cape Town', AVG([Mean TemperatureC]) FROM week06.capetown_cpt_2014
    UNION ALL SELECT 'Delhi',    AVG([Mean TemperatureC]) FROM week06.delhi_del_2014
    UNION ALL SELECT 'London', AVG([Mean TemperatureC]) as Avg_temp FROM week06.london_2014
    UNION ALL SELECT 'Moscow',   AVG([Mean TemperatureC]) FROM week06.moscow_svo_2014
    ORDER BY city
""")

,city,avg_mean_temp_c
0,Beijing,13
1,Brasilia,22
2,Cape Town,17
3,Delhi,13
4,London,12
5,Moscow,5


In [78]:
tabel = q("""
    SELECT TOP 5 *
    FROM week17.loan_approval_prediction
""")

In [81]:
import pandas as pd
tabel['Loan_Approved']

0    1
1    1
2    0
3    1
4    1
Name: Loan_Approved, dtype: int64

### PRACTICE WITH NUMPY

In [166]:
# this code conceals irrelevant warning messages
import warnings
warnings.simplefilter('ignore', FutureWarning)

In [167]:
import numpy as np  
arr = np.array([1, 2, 3, 4, 5])  
print(arr)  
print(type(arr)) 

[1 2 3 4 5]
<class 'numpy.ndarray'>


In [169]:
## Checking NumPy Version  

import numpy as np  
print(np.__version__)  

2.5.3


In [172]:
x = np.array([1,2, 3, 4, 6], ndmin = 10)
print(x.ndim)

10


In [173]:
import numpy as np  
arr = np.array([1, 2, 3, 4, 5, 6, 7])  
print(arr[4:]) 

[5 6 7]


In [174]:
import numpy as np  
arr = np.array([1, 2, 3, 4, 5, 6, 7])  
print(arr[:4])  

[1 2 3 4]


In [175]:
import numpy as np  
arr = np.array([1, 2, 3, 4, 5, 6, 7])  
print(arr[-3:-1])  

[5 6]


In [ ]:
arr = np.array([1, 2, 3, 4, 5, 6, 7])  
print(arr[::2])  

[1 3 5 7]


In [ ]:
arr = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10],  [6, 7, 8, 9, 10]])  
print(arr[1:1]) 

[]


In [ ]:
arr = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]])  
print(arr[0:2, 2]) 

[3 8]


In [95]:
my_arr = np.array([1, 2, 3, 4, 5])
mine = my_arr.copy()
mymy = my_arr.view()
mine[2] = 17
mymy[2] = 57
print(mine.base)
print(mymy.base)
print(my_arr.base)

None
[ 1  2 57  4  5]
None


In [102]:
new_arr = np.array([[1, 2, 3], [0, 5, 6]])
print(new_arr.shape)

(2, 3)


In [116]:
#Practicing Reshaping an Array
b_array = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 3, 4, 5, 6, 7])
new_b_array = b_array.reshape(-1).copy()
print(new_b_array)

[1 2 3 4 5 6 7 8 9 0 3 4 5 6 7]


In [120]:
for x in new_b_array:
    print(x * 10)

10
20
30
40
50
60
70
80
90
0
30
40
50
60
70


In [131]:
new_arr = []
for c in np.nditer(new_b_array):
    print(c)

1
2
3
4
5
6
7
8
9
0
3
4
5
6
7


In [ ]:
arr = np.array([1, 2, 3])  
for x in np.nditer(arr, flags=['buffered'], op_dtypes=['S']):  
    print(x)

np.bytes_(b'1')
np.bytes_(b'2')
np.bytes_(b'3')


In [133]:
for idx, x in np.ndenumerate(new_b_array):
    print(idx, x)

(0,) 1
(1,) 2
(2,) 3
(3,) 4
(4,) 5
(5,) 6
(6,) 7
(7,) 8
(8,) 9
(9,) 0
(10,) 3
(11,) 4
(12,) 5
(13,) 6
(14,) 7


In [142]:
arr = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])
arr = arr.reshape(2, 2, -1)
print(arr.shape, arr.ndim, '\n')

for idx, x in np.ndenumerate(arr):
    print(idx, x)

(2, 2, 2) 3 

(0, 0, 0) 1
(0, 0, 1) 2
(0, 1, 0) 3
(0, 1, 1) 4
(1, 0, 0) 5
(1, 0, 1) 6
(1, 1, 0) 7
(1, 1, 1) 8


In [148]:
arr1 = np.array([[1, 2], 
                [3, 4]])  
arr2 = np.array([[5, 6], 
                [7, 8]])  
arr = np.concatenate((arr1, arr2), axis=1) 
print(arr)

[[1 2 5 6]
 [3 4 7 8]]


In [149]:
arr1 = np.array([1, 2, 3])  
arr2 = np.array([4, 5, 6])  
arr = np.stack((arr1, arr2), axis=1)  
print(arr)

[[1 4]
 [2 5]
 [3 6]]


In [156]:
arr1 = np.array([1, 2, 3])  
arr2 = np.array([4, 5, 6])  
arr = np.dstack((arr1, arr2)) 
print(arr)

[[[1 4]
  [2 5]
  [3 6]]]


In [160]:
a = np.where(arr == 5)
print(a)

(array([0]), array([1]), array([1]))
